# Homework 3  
Sarayu Rao

##1. Building a model

Main Question: What are the best predictors of a driver's finishing position for a given race? 

Data Preparation

In [0]:
#Pyspark Imports
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min, sum, when, regexp_extract
import pyspark.sql.functions as F
import pandas as pd

#ML imports
from sklearn.model_selection import train_test_split 
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
import os
import matplotlib.pyplot as plt
import mlflow.sklearn
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error, explained_variance_score
import tempfile


In [0]:
#Load pitstop dataset
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

#Case necessary columns to integers
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

#Get each driver's average pitstop time for each race
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")


In [0]:
#Load results dataset
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

#Join average pitstop time for each driver's race and rename column
df_race_results = df_results.join(df_avg_pit, on = ["raceId","driverId"])
df_race_results = df_race_results.withColumnRenamed("avg(milliseconds)", "avgPitstop")
display(df_race_results)


In [0]:
#Keep and cast necessary columns to integers for our model
df_race_results = df_race_results.select(["raceId","driverId","resultId","positionOrder","laps","fastestLap","fastestLapTime","fastestLapSpeed","avgPitstop","grid", "rank","fastestLapTime"])

df_race_results = df_race_results.filter((df_race_results["fastestLap"] != "\\N") & (df_race_results["fastestLapTime"] != "\\N") & (df_race_results["fastestLapSpeed"] != "\\N"))
df_race_results = df_race_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                      "laps": col("laps").cast("int"),
                                      "fastestLap": col("fastestLap").cast("int"),
                                      "fastestLapSpeed": col("fastestLapSpeed").cast("float"),
                                      "avgPitstop": col("avgPitstop").cast("float"),
                                      "grid": col("grid").cast("int"),
                                      "rank":col("rank").cast("int"),
                                     })

#Converting fastest lap time to miliseconds 
df_race_results = df_race_results.withColumn("fastestLapTime_ms",
    when(col("fastestLapTime") != "\\N",
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 1).cast("int") * 60000) +  # minutes
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 2).cast("int") * 1000) +  # seconds
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 3).cast("int") * 1)      # milliseconds
    ).otherwise(None)
)
display(df_race_results)

In [0]:
df_pandas = df_race_results.toPandas()
X = df_pandas[['laps', 'fastestLap', 'fastestLapSpeed', 'avgPitstop', 'grid', 'rank', 'fastestLapTime_ms']]
y = df_pandas[['positionOrder']]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

y_train = y_train.values.ravel()
y_test = y_test.values.ravel()


In [0]:
#Running initial flow to get experiementID to use in future log function

with mlflow.start_run(run_name="3rd Initial RF Experiment") as run:
  # Create model, train it, and create predictions
  rf = RandomForestRegressor()
  rf.fit(X_train, y_train)
  #Make sure positionOrder is a whole number between 1 and 30
  predictions = rf.predict(X_test).round().astype(int).clip(1, 30)
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-regressor")
  
  # Create metrics
  mse = mean_squared_error(y_test, predictions)
  mae = mean_absolute_error(y_test, predictions)
  r2 = r2_score(y_test, predictions)
  rmse = root_mean_squared_error(y_test, predictions)
  mape = mean_absolute_percentage_error(y_test, predictions)
  explained_variance = explained_variance_score(y_test, predictions)
  
  print("  mse: {}".format(mse))
  print("  mae: {}".format(mae))
  print("  r2: {}".format(r2))
  print("  rmse: {}".format(rmse))
  print("  mape: {}".format(mape))
  print("  explained_variance: {}".format(explained_variance))
  
  # Log metrics
  mlflow.log_metric("mse", mse)
  mlflow.log_metric("mae", mae)
  mlflow.log_metric("r2", r2)
  mlflow.log_metric("rmse", rmse)
  mlflow.log_metric("mape", mape)
  mlflow.log_metric("explained_variance", explained_variance)

  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
def f1_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    #Make sure positionOrder is a whole number between 1 and 20
    predictions = rf.predict(X_test).round().astype(int).clip(1, 30)

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-regressor")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = root_mean_squared_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    explained_variance = explained_variance_score(y_test, predictions)
    
    print("  mse: {}".format(mse))
    print("  mae: {}".format(mae))
    print("  r2: {}".format(r2))
    print("  rmse: {}".format(rmse))
    print("  mape: {}".format(mape))
    print("  explained_variance: {}".format(explained_variance))
    
    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mape", mape)
    mlflow.log_metric("explained_variance", explained_variance)

    
    # Create feature importance
    importance = pd.DataFrame(list(zip(X.columns, rf.feature_importances_)), 
                                columns=["Feature", "Importance"]
                              ).sort_values("Importance", ascending=False)
    
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close() # Delete the temp file
    
    # Create plot showing actual vs predicted values
    
    fig, ax = plt.subplots()
    ax.scatter(y_test, predictions, alpha=0.5)
    # perfect prediction line
    ax.plot([1, 20], [1, 20], 'r--')  

    ax.set_xlabel("Actual Position")
    ax.set_ylabel("Predicted Position")
    ax.set_title("Actual vs Predicted Position")


    # Log residuals using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="actual_v_predict", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "actual_v_predict.png")
    finally:
      temp.close() # Delete the temp file
      
    display(fig)
    return run.info.run_id

In [0]:
params = {
  "n_estimators": 100,
  "max_depth": 5,
  "random_state": 42
}

f1_rf(experimentID, "5th Testing Flow Outputs", params, X_train, X_test, y_train, y_test)